# ScoutIQ – Model experiments
Two experiments that motivated design decisions. Every number is computed when this notebook runs.

In [1]:
import sys, warnings; sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config as C
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .25})
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from src.preprocessing import build_preprocessor, build_model, feature_sets, TARGET, OFFSET
from src.evaluate import regression_metrics
from sklearn.pipeline import Pipeline
df = pd.read_parquet(C.SNAPSHOTS_PATH)
tr, va, te = (df[df.split==s] for s in ('train','val','test'))
num, cat = feature_sets('A')
hgb = lambda: HistGradientBoostingRegressor(learning_rate=.05, max_iter=400, max_leaf_nodes=31, l2_regularization=1., min_samples_leaf=40, random_state=42)

## Experiment 1 – plain log target vs. market-index-deflated target
Valuations inflate over time. A chronological split turns that drift into systematic bias unless it is modelled with information available before the prediction date.

In [2]:
plain = TransformedTargetRegressor(Pipeline([('prep', build_preprocessor(num, cat, False)), ('m', hgb())]), func=np.log, inverse_func=np.exp)
plain.fit(tr[num+cat], tr[TARGET])
defl = build_model(hgb(), num, cat, False).fit(tr[num+cat+[OFFSET]], tr[TARGET])
rows = []
for name, m, cols in (('plain log target', plain, num+cat), ('deflated log target (used)', defl, num+cat+[OFFSET])):
    for sname, d in (('val', va), ('test', te)):
        pr = m.predict(d[cols]); r = regression_metrics(d[TARGET], pr)
        rows.append(dict(model=name, split=sname, MAE_eur_m=r['MAE_eur']/1e6, R2=r['R2'], MAE_log=r['MAE_log'], median_signed_log_err=float(np.median(np.log(pr)-np.log(d[TARGET])))))
display(pd.DataFrame(rows).round(3))

,model,split,MAE_eur_m,R2,MAE_log,median_signed_log_err
0,plain log target,val,3.033,0.667,0.492,-0.143
1,plain log target,test,3.689,0.636,0.552,-0.275
2,deflated log target (used),val,2.657,0.788,0.493,0.168
3,deflated log target (used),test,3.021,0.793,0.521,0.153


Positive signed error = over-prediction. The bias column shows whether deflation removes the drift.

## Experiment 2 – fundamentals (Model A) vs. adding own valuation history (Model B)
Model B includes the valuation ≥180 days earlier. It is far more accurate but largely learns persistence, so it is *not* appropriate for finding players the market might misprice.

In [3]:
rows = []
for kind in ('A','B'):
    n, c = feature_sets(kind); m = build_model(hgb(), n, c, False).fit(tr[n+c+[OFFSET]], tr[TARGET])
    for sname, d in (('val', va), ('test', te)):
        d2 = d.dropna(subset=['log_lag1_rel']) if kind=='B' else d
        r = regression_metrics(d2[TARGET], m.predict(d2[n+c+[OFFSET]]))
        rows.append(dict(feature_set=kind, split=sname, n=r['n'], MAE_eur_m=r['MAE_eur']/1e6, R2=r['R2'], MedianAPE=r['MedianAPE_pct']))
display(pd.DataFrame(rows).round(3))
print('NB: Model B rows require a prior valuation, so its evaluation set is smaller.')

,feature_set,split,n,MAE_eur_m,R2,MedianAPE
0,A,val,10076,2.657,0.788,42.249
1,A,test,9784,3.021,0.793,43.210
2,B,val,10058,1.362,0.935,18.249
3,B,test,9737,1.769,0.905,21.386


NB: Model B rows require a prior valuation, so its evaluation set is smaller.


## Experiment 3 – final comparison table (from `outputs/metrics`)

In [4]:
display(pd.read_csv(C.METRICS_DIR/'model_comparison.csv').round(3).iloc[:, [0,1,2,4,6,15,17,19,20]])

,feature_set,model,config,val_MAE_eur,val_R2,test_R2,test_MedianAPE_pct,test_RMSLE,test_R2_log
0,A,Dummy (median),median,5989306.305,-0.128,-0.142,86.111,1.570,-0.010
1,A,Linear Regression,ols,3099206.575,0.627,0.657,46.698,0.691,0.804
2,A,Ridge,alpha=10,3094089.529,0.629,0.662,46.865,0.691,0.804
3,A,Random Forest,"n=200,leaf=5,mf=0.4",2908473.319,0.713,0.731,46.280,0.704,0.797
4,A,HistGradientBoosting,"lr=0.05,iter=400,leaf=31",2656896.857,0.788,0.793,43.210,0.662,0.820
5,B,Ridge,alpha=100,1535341.131,0.862,0.858,22.203,0.407,0.932
6,B,HistGradientBoosting,"lr=0.05,iter=400,leaf=31",1360801.363,0.935,0.905,21.440,0.377,0.942
